# VoxCPM-0.5B — LoRA Fine-Tuning on Custom Voice Data
**Runtime:** Colab T4 (16 GB VRAM)  
**Output:** one LoRA adapter per speaker, hot-swappable in `voice-agent-api`

## 1. Check GPU

In [ ]:
import subprocess, sys
result = subprocess.run(["nvidia-smi","--query-gpu=name,memory.total","--format=csv,noheader"],
                        capture_output=True, text=True)
print(result.stdout.strip() or "No GPU — switch runtime to T4")
import torch
print(f"PyTorch {torch.__version__} | CUDA: {torch.cuda.is_available()}")


## 2. Install dependencies

In [ ]:
%pip install -q voxcpm faster-whisper pydub soundfile librosa tensorboardX argbind
!apt-get install -qq -y ffmpeg > /dev/null 2>&1
print("Done")


## 3. Clone VoxCPM repo (contains training scripts)

In [ ]:
import os
REPO_DIR = "/content/VoxCPM"
if not os.path.exists(REPO_DIR):
    !git clone --depth 1 https://github.com/OpenBMB/VoxCPM {REPO_DIR}
os.chdir(REPO_DIR)
!ls scripts/


## 4. Mount Drive & define paths

In [ ]:
USE_DRIVE = True   # set False to stay in /content
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE_DIR = "/content/drive/MyDrive/voice_clone_project"
else:
    BASE_DIR = "/content/voice_clone_project"

import os
RAW_DIR      = f"{BASE_DIR}/raw_voices"   # raw_voices/<speaker>/*.wav|mp3
PREPARED_DIR = f"{BASE_DIR}/prepared"
MANIFEST_DIR = f"{BASE_DIR}/manifests"
CKPT_DIR     = f"{BASE_DIR}/checkpoints"
LOG_DIR      = f"{BASE_DIR}/logs"

for d in [RAW_DIR, PREPARED_DIR, MANIFEST_DIR, CKPT_DIR, LOG_DIR]:
    os.makedirs(d, exist_ok=True)
print(f"BASE_DIR = {BASE_DIR}")


## 5. Upload voice clips
**Structure:** `raw_voices/<speaker_name>/*.wav|mp3|m4a|flac`  
Run this cell once per speaker, or upload directly to Drive and skip.

In [ ]:
from google.colab import files
import os
SPEAKER_NAME = "shubham"   # change per speaker
speaker_dir = f"{RAW_DIR}/{SPEAKER_NAME}"
os.makedirs(speaker_dir, exist_ok=True)
uploaded = files.upload()
for fn, data in uploaded.items():
    with open(f"{speaker_dir}/{fn}", "wb") as f:
        f.write(data)
    print(f"Saved {speaker_dir}/{fn}")


## 6. Data preparation — convert, trim silence, transcribe

In [ ]:
import os, random, json
from pathlib import Path
from pydub import AudioSegment
from pydub.silence import detect_nonsilent
import soundfile as sf
from faster_whisper import WhisperModel

TARGET_SR        = 16000
MIN_CLIP_SECS    = 2.0
MAX_CLIP_SECS    = 25.0
WHISPER_MODEL    = "base"
WHISPER_LANGUAGE = "en"   # None = auto-detect
REF_AUDIO_RATIO  = 0.4
TRAIN_SPLIT      = 0.9
AUDIO_EXTS       = {".wav", ".mp3", ".m4a", ".flac", ".ogg", ".webm"}

print(f"Loading Whisper {WHISPER_MODEL} ...")
asr = WhisperModel(WHISPER_MODEL, device="cuda", compute_type="float16")
print("Whisper loaded")


def convert_clip(src: Path, dst: Path):
    try:
        audio = AudioSegment.from_file(src)
    except Exception as e:
        print(f"  Cannot decode {src.name}: {e}")
        return None
    audio = audio.set_channels(1).set_frame_rate(TARGET_SR).set_sample_width(2)
    nonsilent = detect_nonsilent(audio, min_silence_len=200, silence_thresh=-40)
    if nonsilent:
        audio = audio[nonsilent[0][0]:nonsilent[-1][1]]
    dur = len(audio) / 1000
    if not (MIN_CLIP_SECS <= dur <= MAX_CLIP_SECS):
        print(f"  Skip {src.name}: {dur:.1f}s out of range")
        return None
    dst.parent.mkdir(parents=True, exist_ok=True)
    audio.export(dst, format="wav")
    return dur


def transcribe(wav: Path) -> str:
    segs, _ = asr.transcribe(str(wav), language=WHISPER_LANGUAGE, beam_size=5)
    return " ".join(s.text.strip() for s in segs).strip()


def build_manifest(speaker, wavs, train_out, val_out):
    random.shuffle(wavs)
    split = max(1, int(len(wavs) * TRAIN_SPLIT))
    parts = {"train": wavs[:split], "val": wavs[split:]}
    total = 0
    for name, paths in parts.items():
        if not paths:
            continue
        out = train_out if name == "train" else val_out
        rows = []
        for wp in paths:
            text = transcribe(wp)
            if not text:
                continue
            row = {"audio": str(wp), "text": text}
            if random.random() < REF_AUDIO_RATIO:
                cands = [p for p in wavs if p != wp]
                if cands:
                    ref = random.choice(cands)
                    ref_segs, _ = asr.transcribe(str(ref), language=WHISPER_LANGUAGE, beam_size=5)
                    row["ref_audio"] = str(ref)
                    row["ref_text"] = " ".join(s.text.strip() for s in ref_segs).strip()
            rows.append(row)
            total += 1
            print(f"  [{name}] {wp.name}: {text[:70]}")
        with open(out, "w") as f:
            for r in rows:
                f.write(json.dumps(r, ensure_ascii=False) + "\n")
        print(f"  -> {out} ({len(rows)} samples)")
    return total


speakers = [d for d in Path(RAW_DIR).iterdir() if d.is_dir()]
print(f"Speakers: {[s.name for s in speakers]}")

for spk_dir in speakers:
    speaker = spk_dir.name
    print(f"\n=== {speaker} ===")
    prepared = Path(PREPARED_DIR) / speaker
    prepared.mkdir(parents=True, exist_ok=True)
    valid = []
    for src in sorted(spk_dir.iterdir()):
        if src.suffix.lower() not in AUDIO_EXTS:
            continue
        dst = prepared / (src.stem + ".wav")
        dur = convert_clip(src, dst)
        if dur:
            valid.append(dst)
            print(f"  OK {src.name} ({dur:.2f}s)")
    if not valid:
        print(f"  No valid clips for {speaker}")
        continue
    train_jl = Path(MANIFEST_DIR) / f"{speaker}_train.jsonl"
    val_jl   = Path(MANIFEST_DIR) / f"{speaker}_val.jsonl"
    n = build_manifest(speaker, valid, train_jl, val_jl)
    print(f"  {speaker}: {n} samples")

print("\nAll speakers processed")


## 7. Validate manifests

In [ ]:
import os
from pathlib import Path
for mf in sorted(Path(MANIFEST_DIR).glob("*.jsonl")):
    out = os.popen(f"voxcpm validate --manifest {mf} 2>&1").read()
    ok = "error" not in out.lower()
    print(("OK" if ok else "FAIL"), mf.name)
    if not ok:
        print("   ", out.strip())


## 8. Generate per-speaker LoRA configs

In [ ]:
import yaml
from pathlib import Path

BATCH_SIZE       = 4
GRAD_ACCUM       = 4      # effective batch = 16
MAX_BATCH_TOKENS = 4096
LORA_RANK        = 32
LORA_ALPHA       = 32
LORA_DROPOUT     = 0.05
LR               = 1e-4


def count_manifest(p: Path) -> int:
    with open(p) as f:
        return sum(1 for _ in f)


def write_config(speaker, train_jl, val_jl) -> str:
    n = count_manifest(train_jl)
    iters    = max(30, min(800, int(3 * n / BATCH_SIZE)))
    save_int = max(10, iters // 5)
    warmup   = max(5,  iters // 10)
    cfg = {
        "model": {
            "pretrained_path": "openbmb/VoxCPM-0.5B",
            "sample_rate": 16000,
        },
        "lora": {
            "enabled": True,
            "r": LORA_RANK,
            "alpha": LORA_ALPHA,
            "dropout": LORA_DROPOUT,
            "target_modules": [
                "q_proj","k_proj","v_proj","o_proj",
                "gate_proj","up_proj","down_proj",
            ],
        },
        "train": {
            "train_manifest": str(train_jl),
            "val_manifest": str(val_jl) if Path(val_jl).exists() else str(train_jl),
            "batch_size": BATCH_SIZE,
            "grad_accum_steps": GRAD_ACCUM,
            "max_batch_tokens": MAX_BATCH_TOKENS,
            "num_iters": iters,
            "warmup_steps": warmup,
            "learning_rate": LR,
            "log_interval": max(5, iters // 20),
            "save_interval": save_int,
            "valid_interval": save_int,
            "save_path": f"{CKPT_DIR}/{speaker}",
            "tb_log_dir": f"{LOG_DIR}/{speaker}",
        },
    }
    cfg_path = str(Path(MANIFEST_DIR) / f"{speaker}_lora_config.yaml")
    with open(cfg_path, "w") as f:
        yaml.dump(cfg, f, default_flow_style=False)
    print(f"  {speaker}: {n} samples, {iters} iters -> {cfg_path}")
    return cfg_path


speaker_configs = {}
for spk_dir in sorted(Path(PREPARED_DIR).iterdir()):
    if not spk_dir.is_dir():
        continue
    s = spk_dir.name
    train_jl = Path(MANIFEST_DIR) / f"{s}_train.jsonl"
    val_jl   = Path(MANIFEST_DIR) / f"{s}_val.jsonl"
    if not train_jl.exists():
        continue
    speaker_configs[s] = write_config(s, train_jl, val_jl)

print("Configs:", list(speaker_configs.keys()))


## 9. Train — one LoRA per speaker  
~10–30 min per speaker on T4, depending on dataset size.

In [ ]:
import subprocess, sys

TRAIN_SCRIPT = f"{REPO_DIR}/scripts/train_voxcpm_finetune.py"

for speaker, cfg_path in speaker_configs.items():
    print(f"\n=== Training: {speaker} ===")
    r = subprocess.run(
        [sys.executable, TRAIN_SCRIPT, "--config_path", cfg_path],
        cwd=REPO_DIR,
    )
    print("OK" if r.returncode == 0 else f"FAILED (exit {r.returncode})")

print("\nAll done")


## 10. Inference test

In [ ]:
import soundfile as sf
from IPython.display import Audio, display
from voxcpm import VoxCPM
from pathlib import Path

TEST_SPEAKER = list(speaker_configs.keys())[0]   # change as needed
TEST_TEXT    = "Hello, this is a test of my cloned voice."

ckpt_base = Path(CKPT_DIR) / TEST_SPEAKER
checkpoints = sorted(ckpt_base.glob("*/"), key=lambda p: p.stat().st_mtime) if ckpt_base.exists() else []
if not checkpoints:
    print("No checkpoint found. Did training complete?")
else:
    lora_ckpt = str(checkpoints[-1])
    print(f"Loading LoRA: {lora_ckpt}")
    model = VoxCPM.from_pretrained("openbmb/VoxCPM-0.5B", lora_weights_path=lora_ckpt)
    wav = model.generate(
        text=TEST_TEXT,
        cfg_value=2.0,
        inference_timesteps=10,
        normalize=True,
        denoise=True,
        retry_badcase=True,
    )
    out = f"/content/test_{TEST_SPEAKER}.wav"
    sf.write(out, wav, 16000)
    print(f"Saved: {out}")
    display(Audio(out))


## 11. Export — zip checkpoints, upload to voice-agent-api

In [ ]:
import shutil, os
from pathlib import Path
from google.colab import files as colab_files

EXPORT_DIR = f"{BASE_DIR}/export"
os.makedirs(EXPORT_DIR, exist_ok=True)

for speaker in speaker_configs:
    ckpt_base = Path(CKPT_DIR) / speaker
    checkpoints = sorted(ckpt_base.glob("*/"), key=lambda p: p.stat().st_mtime)
    if not checkpoints:
        print(f"No checkpoint for {speaker}")
        continue
    latest = checkpoints[-1]
    zip_base = f"{EXPORT_DIR}/{speaker}_lora"
    shutil.make_archive(zip_base, "zip", root_dir=str(latest.parent), base_dir=latest.name)
    final_zip = zip_base + ".zip"
    print(f"{speaker} -> {final_zip}")
    colab_files.download(final_zip)

print("Upload each zip via POST /voices/{voice_id}/lora")


## 12. Connect to voice-agent-api

```bash
# 1. Create voice profile
curl -X POST http://localhost:8000/voices \
  -F 'name=shubham' -F 'audio_file=@ref_clip.wav'

# 2. Attach LoRA
curl -X POST http://localhost:8000/voices/{voice_id}/lora \
  -F 'lora_zip=@shubham_lora.zip'

# 3. Generate (LoRA used automatically; no reference audio needed)
curl -X POST http://localhost:8000/tts \
  -H 'Content-Type: application/json' \
  -d '{"voice_id": "{voice_id}", "text": "Hello from my cloned voice"}' \
  --output out.wav

# 4. Detach LoRA (revert to zero-shot)
curl -X DELETE http://localhost:8000/voices/{voice_id}/lora
```